# Assignment 6: BLEU Improvement Experiments for my -> ph PBSMT

ဒီ notebook က original baseline ကိုမထိဘဲ **workspace အသစ်** ထဲမှာ Moses/PBSMT experiments တွေ run လုပ်ဖို့ပါ။ Target က `my -> ph` direction အတွက် BLEU score တက်အောင် data normalization, OOV check, LM order, phrase length, MERT tuning settings တွေကိုစမ်းတာပါ။


In [ ]:
%cd /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par


## 1. Setup

Moses/GIZA path နဲ့ output workspace ကိုသတ်မှတ်ပါမယ်။ Output အားလုံး `bleu_improve_workspace` အောက်မှာပဲသိမ်းပါမယ်။


In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
from datetime import datetime

SRC = "my"
TGT = "ph"
PAIR = f"{SRC}-{TGT}"

MOSES = Path("/home/phantom/mosesdecoder")
GIZA = Path("/home/phantom/giza-pp")
DATA_DIR = Path.cwd()
WORK = DATA_DIR / "bleu_improve_workspace"
RAW = WORK / "raw"
NORM = WORK / "normalized"
TOOLS = WORK / "tools"
SUMMARY = WORK / "summary.tsv"

for d in [WORK, RAW, NORM, TOOLS]:
    d.mkdir(parents=True, exist_ok=True)

print("Direction:", f"{SRC} -> {TGT}")
print("Data dir:", DATA_DIR)
print("Workspace:", WORK)
print("Moses:", MOSES)
print("GIZA++:", GIZA)


In [ ]:
# GIZA++ tools ကို workspace ထဲ symlink ချိတ်ထားပါမယ်။
links = {
    TOOLS / "GIZA++": GIZA / "GIZA++-v2/GIZA++",
    TOOLS / "plain2snt.out": GIZA / "GIZA++-v2/plain2snt.out",
    TOOLS / "snt2cooc.out": GIZA / "GIZA++-v2/snt2cooc.out",
    TOOLS / "mkcls": GIZA / "mkcls-v2/mkcls",
}
for dst, src in links.items():
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src)

!ls -l {TOOLS}


## 2. Copy Data Into New Workspace

Original train/dev/test files ကို workspace ထဲ copy လုပ်ထားပါမယ်။ Experiments တွေက ဒီ copy တွေကိုပဲသုံးပါမယ်။


In [ ]:
for split in ["train", "dev", "test"]:
    for lang in [SRC, TGT]:
        src = DATA_DIR / f"{split}.{lang}"
        dst = RAW / f"{split}.{lang}"
        shutil.copy2(src, dst)

!wc -l {RAW}/train.{SRC} {RAW}/train.{TGT} {RAW}/dev.{SRC} {RAW}/dev.{TGT} {RAW}/test.{SRC} {RAW}/test.{TGT}


## 3. Baseline Score

အရင် notebook က baseline result ကို comparison အတွက်သိမ်းထားပါမယ်။ Disk ထဲက current baseline BLEU က `69.59` ဖြစ်ပါတယ်။


In [ ]:
# This cell is self-contained, so it still works even if the setup cell was not run.
from pathlib import Path

if "DATA_DIR" not in globals():
    DATA_DIR = Path("/home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par")
if "WORK" not in globals():
    WORK = DATA_DIR / "bleu_improve_workspace"

WORK.mkdir(parents=True, exist_ok=True)

baseline_bleu_file = DATA_DIR / "work/my-ph/decode/test.multi-bleu.my-ph.txt"
if baseline_bleu_file.exists():
    baseline_text = baseline_bleu_file.read_text(encoding="utf-8").strip()
else:
    baseline_text = "BLEU = 69.59, 85.2/72.6/64.5/58.8 (BP=1.000, ratio=1.000, hyp_len=8049, ref_len=8048)"

(WORK / "baseline.multi-bleu.txt").write_text(baseline_text + "\n", encoding="utf-8")
print(baseline_text)


## 4. Normalize Data

BLEU တက်ဖို့ ပထမဆုံးလုပ်သင့်တာက train/dev/test အားလုံး spacing နဲ့ Unicode normalization consistent ဖြစ်အောင်လုပ်တာပါ။ ဒီ step က raw data ကိုမပြင်ဘဲ `normalized/` folder ထဲမှာ version အသစ်ထုတ်ပါမယ်။


In [ ]:
ZERO_WIDTH = dict.fromkeys(map(ord, "\u200b\u200c\u200d\ufeff"), None)

try:
    import unicodedata
except ImportError:
    unicodedata = None

def normalize_line(line: str) -> str:
    line = line.translate(ZERO_WIDTH)
    if unicodedata:
        line = unicodedata.normalize("NFC", line)
    line = line.replace("‘", "'").replace("’", "'").replace("`", "'")
    line = line.replace("“", '"').replace("”", '"')
    line = re.sub(r"\s+", " ", line).strip()
    return line

for split in ["train", "dev", "test"]:
    for lang in [SRC, TGT]:
        inp = RAW / f"{split}.{lang}"
        out = NORM / f"{split}.{lang}"
        with inp.open(encoding="utf-8") as fi, out.open("w", encoding="utf-8") as fo:
            for line in fi:
                fo.write(normalize_line(line) + "\n")

!wc -l {NORM}/train.{SRC} {NORM}/train.{TGT} {NORM}/dev.{SRC} {NORM}/dev.{TGT} {NORM}/test.{SRC} {NORM}/test.{TGT}
!head -n 5 {NORM}/train.{SRC}
!head -n 5 {NORM}/train.{TGT}


## 5. OOV Diagnostic

Test source ထဲက training source မှာမပါတဲ့ token တွေကိုစစ်ပါမယ်။ OOV များရင် decoded output ထဲမှာ `UNK` တွေထွက်ပြီး BLEU ကျနိုင်ပါတယ်။


In [ ]:
def vocab(path):
    words = set()
    with Path(path).open(encoding="utf-8") as f:
        for line in f:
            words.update(line.split())
    return words

train_vocab = vocab(NORM / f"train.{SRC}")
test_vocab = vocab(NORM / f"test.{SRC}")
oov = sorted(test_vocab - train_vocab)

(WORK / "oov_tokens.txt").write_text("\n".join(oov) + ("\n" if oov else ""), encoding="utf-8")
print("train vocab:", len(train_vocab))
print("test vocab:", len(test_vocab))
print("OOV count:", len(oov))
print("first 50 OOV:", oov[:50])


## 6. Experiment Runner

ဒီ function က experiment တစ်ခုလုံးကို run ပေးပါတယ်။

- target LM ဆောက်မယ်
- phrase-based SMT train မယ် (`-f my -e ph`)
- MERT tuning လုပ်မယ်
- test decode လုပ်မယ်
- BLEU score တွက်မယ်
- summary.tsv ထဲသိမ်းမယ်

တူညီတဲ့ experiment result ရှိပြီးသားဆိုရင် default အနေနဲ့ cached result ကိုပဲပြန်သုံးပါတယ်။ အသစ်ပြန် run ချင်ရင် `force=True` ထည့်ပါ။


In [ ]:
def run_cmd(cmd, cwd=DATA_DIR, log_path=None):
    print("\n$", " ".join(map(str, cmd)))
    start = datetime.now()
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    elapsed = datetime.now() - start
    output = proc.stdout
    if log_path:
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        Path(log_path).write_text(output, encoding="utf-8")
    print(output[-4000:])
    print("elapsed:", elapsed)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(map(str, cmd))}")
    return output

def parse_bleu(text):
    m = re.search(r"BLEU = ([0-9.]+)", text)
    return float(m.group(1)) if m else None

def append_summary(row):
    header = list(row.keys())
    old_rows = []
    if SUMMARY.exists():
        with SUMMARY.open(encoding="utf-8") as f:
            rows = [line.rstrip("\n").split("\t") for line in f if line.strip()]
        if rows:
            old_rows = [r for r in rows[1:] if r and r[0] != row["name"]]
    with SUMMARY.open("w", encoding="utf-8") as f:
        f.write("\t".join(header) + "\n")
        for r in old_rows:
            f.write("\t".join(r) + "\n")
        f.write("\t".join(str(row[h]) for h in header) + "\n")

def run_experiment(name, data_dir=NORM, lm_order=5, max_phrase_length=7,
                   alignment="grow-diag-final-and", random_restarts=20,
                   maximum_iterations=25, cores=4, force=False):
    exp = WORK / "experiments" / name
    lm_dir = exp / "lm"
    model_dir = exp / "model"
    mert_dir = exp / "mert"
    decode_dir = exp / "decode"
    for d in [lm_dir, model_dir, mert_dir, decode_dir]:
        d.mkdir(parents=True, exist_ok=True)

    decoded = decode_dir / f"test.decoded.{TGT}"
    bleu_file = decode_dir / "test.multi-bleu.txt"
    if bleu_file.exists() and decoded.exists() and not force:
        bleu_text = bleu_file.read_text(encoding="utf-8")
        row = {
            "name": name,
            "bleu": parse_bleu(bleu_text),
            "lm_order": lm_order,
            "max_phrase_length": max_phrase_length,
            "alignment": alignment,
            "random_restarts": random_restarts,
            "maximum_iterations": maximum_iterations,
            "decoded": str(decoded),
            "bleu_file": str(bleu_file),
        }
        append_summary(row)
        print(f"Using cached result for {name}")
        print(bleu_text)
        return row

    train_prefix = data_dir / "train"
    dev_src = data_dir / f"dev.{SRC}"
    dev_tgt = data_dir / f"dev.{TGT}"
    test_src = data_dir / f"test.{SRC}"
    test_tgt = data_dir / f"test.{TGT}"

    arpa = lm_dir / f"train.{TGT}.{lm_order}gram.arpa"
    blm = lm_dir / f"train.{TGT}.{lm_order}gram.blm"

    if force or not blm.exists():
        run_cmd([MOSES / "bin/lmplz", "-o", lm_order, "--text", data_dir / f"train.{TGT}", "--arpa", arpa], log_path=exp / "lmplz.log")
        run_cmd([MOSES / "bin/build_binary", arpa, blm], log_path=exp / "build_binary.log")

    if force or not (model_dir / "moses.ini").exists():
        train_cmd = [
            "perl", MOSES / "scripts/training/train-model.perl",
            "-root-dir", exp,
            "-corpus", train_prefix,
            "-f", SRC, "-e", TGT,
            "-alignment", alignment,
            "-reordering", "msd-bidirectional-fe",
            "-max-phrase-length", max_phrase_length,
            "-lm", f"0:{lm_order}:{blm}:8",
            "-external-bin-dir", TOOLS,
            "-cores", cores,
        ]
        run_cmd(train_cmd, log_path=exp / "training.log")

    if force or not (mert_dir / "moses.ini").exists():
        mert_cmd = [
            "perl", MOSES / "scripts/training/mert-moses.pl",
            dev_src, dev_tgt,
            MOSES / "bin/moses",
            model_dir / "moses.ini",
            "--mertdir", MOSES / "bin",
            "--working-dir", mert_dir,
            "--decoder-flags", f"-threads {cores}",
            "--random-restarts", random_restarts,
            "--maximum-iterations", maximum_iterations,
            "--return-best-dev",
            "--predictable-seeds",
        ]
        run_cmd(mert_cmd, log_path=exp / "mert.log")

    decode_cmd = [
        MOSES / "bin/moses",
        "-f", mert_dir / "moses.ini",
        "-threads", cores,
        "-input-file", test_src,
    ]
    print("\n$", " ".join(map(str, decode_cmd)))
    decode_proc = subprocess.run(
        [str(x) for x in decode_cmd],
        cwd=str(DATA_DIR),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    decoded_text = decode_proc.stdout
    decoded.write_text(decoded_text, encoding="utf-8")
    (decode_dir / "decode.log").write_text(decode_proc.stderr, encoding="utf-8")
    print(decode_proc.stderr[-2000:])
    if decode_proc.returncode != 0:
        raise RuntimeError("Decode command failed")

    bleu_proc = subprocess.run(
        ["perl", str(MOSES / "scripts/generic/multi-bleu.perl"), str(test_tgt)],
        cwd=str(DATA_DIR),
        input=decoded_text,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    bleu_text = bleu_proc.stdout
    bleu_file.write_text(bleu_text, encoding="utf-8")
    print(bleu_text)
    if bleu_proc.returncode != 0:
        raise RuntimeError("BLEU command failed")

    row = {
        "name": name,
        "bleu": parse_bleu(bleu_text),
        "lm_order": lm_order,
        "max_phrase_length": max_phrase_length,
        "alignment": alignment,
        "random_restarts": random_restarts,
        "maximum_iterations": maximum_iterations,
        "decoded": str(decoded),
        "bleu_file": str(bleu_file),
    }
    append_summary(row)
    return row


## 7. Recommended Experiments

အောက်က experiments တွေကို တစ်ခုချင်း run ပြီး BLEU ကို baseline `69.59` နဲ့နှိုင်းယှဉ်ပါ။ Run time အနည်းငယ်ကြာနိုင်ပါတယ်။ ပထမဆုံး `norm_5g_p7_rr30` ကို run လုပ်ပါ။


In [ ]:
# Experiment 1: normalized data + stronger MERT random restarts
# Expected impact: tuning stability ပိုကောင်းနိုင်ပါတယ်။
exp1 = run_experiment(
    name="norm_5g_p7_rr30",
    data_dir=NORM,
    lm_order=5,
    max_phrase_length=7,
    alignment="grow-diag-final-and",
    random_restarts=30,
    maximum_iterations=35,
    cores=4,
)
exp1


In [ ]:
# Experiment 2: higher-order target LM (6-gram fallback)
# ဘာကြောင့်ပြင်လဲ:
#   မူလစမ်းချင်တာက 7-gram ပါ။ ဒါပေမဲ့ ဒီစက်ထဲက KenLM build က max order 6 အထိပဲ support လုပ်လို့
#   7-gram build_binary မှာ "This model has order 7 but KenLM was compiled to support up to 6" error တက်ပါတယ်။
#   ဒါကြောင့် support လုပ်တဲ့အမြင့်ဆုံး LM order ဖြစ်တဲ့ 6-gram ကို fallback အနေနဲ့စမ်းပါတယ်။
# ဘယ်နေရာပြင်လဲ:
#   lm_order=5 ကနေ lm_order=6 ပြောင်းထားပါတယ်။
#   train-model.perl ထဲက -lm 0:6:<6gram.blm>:8 ဖြစ်သွားပါမယ်။
exp2 = run_experiment(
    name="norm_6g_p7_rr30",
    data_dir=NORM,
    lm_order=6,
    max_phrase_length=7,
    alignment="grow-diag-final-and",
    random_restarts=30,
    maximum_iterations=35,
    cores=4,
)
exp2


In [ ]:
# Experiment 3: shorter phrase length
# ဘာကြောင့်ပြင်လဲ:
#   G2P မှာ source syllable/grapheme chunk နဲ့ target ph chunk တွေ တိုတိုလေးတွဲတာက
#   တခါတလေ phrase table ကိုပိုတိကျစေနိုင်ပါတယ်။
# ဘယ်နေရာပြင်လဲ:
#   max_phrase_length=7 ကနေ max_phrase_length=5 ပြောင်းထားပါတယ်။
#   train-model.perl ထဲက -max-phrase-length 5 ဖြစ်သွားပါမယ်။
exp3 = run_experiment(
    name="norm_5g_p5_rr30",
    data_dir=NORM,
    lm_order=5,
    max_phrase_length=5,
    alignment="grow-diag-final-and",
    random_restarts=30,
    maximum_iterations=35,
    cores=4,
)
exp3


In [ ]:
# Experiment 4: stricter alignment
# ဘာကြောင့်ပြင်လဲ:
#   grow-diag-final-and က alignment coverage များပေမဲ့ noisy phrase pairs ပါနိုင်ပါတယ်။
#   intersect က stricter alignment ဖြစ်လို့ precision တက်နိုင်မလား စမ်းတာပါ။
# ဘယ်နေရာပြင်လဲ:
#   alignment="grow-diag-final-and" ကနေ alignment="intersect" ပြောင်းထားပါတယ်။
#   train-model.perl ထဲက -alignment intersect ဖြစ်သွားပါမယ်။
exp4 = run_experiment(
    name="norm_5g_p7_intersect_rr30",
    data_dir=NORM,
    lm_order=5,
    max_phrase_length=7,
    alignment="intersect",
    random_restarts=30,
    maximum_iterations=35,
    cores=4,
)
exp4


## 8. Compare Results

Experiment အားလုံးပြီးရင် summary table ကို BLEU အမြင့်ဆုံးအလိုက်ကြည့်ပါ။


In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print

if SUMMARY.exists():
    if pd:
        df = pd.read_csv(SUMMARY, sep="\t")
        display(df.sort_values("bleu", ascending=False))
    else:
        print(SUMMARY.read_text(encoding="utf-8"))
else:
    print("No experiments have been run yet.")

print("Baseline BLEU: 69.59")


## 9. Check Best Output

အကောင်းဆုံး experiment ရလာရင် decoded output နဲ့ BLEU file path ကို summary table ထဲကနေကြည့်နိုင်ပါတယ်။ `UNK` များနေသေးရင် `oov_tokens.txt` ကိုစစ်ပြီး train data ထပ်ဖြည့်တာက tuning ထက်ပိုထိရောက်နိုင်ပါတယ်။


In [ ]:
if SUMMARY.exists():
    try:
        import pandas as pd
    except ImportError:
        pd = None

    if pd:
        df = pd.read_csv(SUMMARY, sep="\t")
        best = df.sort_values("bleu", ascending=False).iloc[0]
        best_name = best["name"]
        best_bleu = best["bleu"]
        best_decoded = Path(best["decoded"])
        best_bleu_file = Path(best["bleu_file"])
    else:
        rows = [line.rstrip("\n").split("\t") for line in SUMMARY.read_text(encoding="utf-8").splitlines()]
        header, data = rows[0], rows[1:]
        best_row = max(data, key=lambda r: float(r[header.index("bleu")]))
        best_name = best_row[header.index("name")]
        best_bleu = best_row[header.index("bleu")]
        best_decoded = Path(best_row[header.index("decoded")])
        best_bleu_file = Path(best_row[header.index("bleu_file")])

    print("Best experiment:", best_name)
    print("Best BLEU:", best_bleu)
    print("Decoded output:", best_decoded)
    print("BLEU file:", best_bleu_file)
    print("\nBLEU detail:")
    print(best_bleu_file.read_text(encoding="utf-8"))
    print("\nFirst 10 decoded lines:")
    print("".join(best_decoded.read_text(encoding="utf-8").splitlines(True)[:10]))
